In [ ]:
import cartopy
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xarray as xr

from project.utils.data import enrich, load_hillshade
from project.utils.paths import get_path
from project.utils.plotting import ProjectCustomExtent, cbar_halo

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr(get_path('predictions_file')).sel(
    time=slice('1993', None))

# Load HRU polygons.
hrus = gpd.read_file(
    get_path('hru_file'), use_arrow=True).set_index('EZGNR')

prevah_hrus = ds.hru.where(ds.q_prevah_mmd.isel(time=0).notnull().compute(), drop=True).values

common_hrus = hrus.loc[hrus.origin == 'ezg'].index.intersection(prevah_hrus)

ds = ds.sel(hru=common_hrus)

ds['R_pred_mean'] = ds['R_pred'].mean('time').compute()
hrus = hrus.loc[common_hrus]

rivers = gpd.read_file(get_path('river_file'), use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

bg = load_hillshade(
    hillshade_dir='./data',
    dem_path=get_path('raw_data_root') / 'raw/merit.zarr',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    force_recompute=False)

In [ ]:
ds_pretrain = xr.open_zarr(get_path('log_root') / 'prevah/xval/fold_000/preds.zarr').sel(
    time=slice('1993', None)).sel(hru=common_hrus)

# Runoff plausibility check

In [ ]:
q_prevah_mean = ds_pretrain.q_prevah_mmd.mean("time").compute()

In [ ]:
q_pretraining_mean = ds_pretrain.q_pred_mmd.mean("time").compute()

In [ ]:
q_mean = ds.q_pred_mmd.mean("time").compute()

In [ ]:
p_mean = ds.prec.mean("time").compute()

# New analysis

In [ ]:
hrus_lakes = hrus.loc[ds.hru.where(ds.is_lake.load() == 1, drop=True)]
not_lake = ds.is_lake.load() == 0
ds = ds.where(not_lake, drop=True)
ds_pretrain = ds_pretrain.where(not_lake, drop=True)


corr_rr_prevah = xr.corr(ds.q_prevah_mmd, ds.q_pred_mmd.sel(model='routed_random'), dim='time').compute()
corr_rr_pretraining = xr.corr(ds_pretrain.q_pred_mmd, ds.q_pred_mmd.sel(model='routed_random'), dim='time').compute()

bias_rr_prevah = (
    ds.q_pred_mmd.sel(model='routed_random').mean('time') - ds.q_prevah_mmd.mean('time')).compute()
bias_rr_pretraining = (
    ds.q_pred_mmd.sel(model='routed_random').mean('time') - ds_pretrain.q_pred_mmd.mean('time')).compute()

q_factor_prevah = q_prevah_mean / p_mean
q_factor_pretraining = q_pretraining_mean / p_mean
q_factor_pred = q_mean.sel(model='routed_random') / p_mean

common_hrus = hrus.loc[hrus.origin == 'ezg'].index.intersection(ds_pretrain.hru.values)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 6), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.01, 'hspace': 0.1})

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

for i, (rr, title) in enumerate(
        zip(
            [corr_rr_pretraining, bias_rr_pretraining],
            ['Runoff Correlation Pre- vs. Post-finetuning', 'Runoff Bias Pre- vs. Post-finetuning'])
    ):

    ax = axes[0, i]
    ax.set_global()

    if i ==0:
        norm = mpl.colors.Normalize(vmin=0.85, vmax=1.0)
        cmap = sns.color_palette('rocket', as_cmap=True)
        label = 'Correlation (-)'
        extend = 'min'
    else:
        norm = mpl.colors.Normalize(vmin=-1, vmax=1)
        # cmap = sns.color_palette("vlag", as_cmap=True)
        cmap = sns.color_palette('PiYG', as_cmap=True)
        label = 'Bias (mm d$^{-1}$)' 
        extend = 'both'

    axins = ax.inset_axes((0.1, 0.88, 0.25, 0.03))
    rr_subset = rr.sel(hru=hrus.loc[common_hrus].index).load()
    im = enrich(
        ds=rr_subset,
        hrus=hrus,
        variables='val'
    ).plot(
        column='val',
        legend=True,
        ec='k',
        lw=0.1,
        alpha=0.6,
        ax=ax,
        norm=norm,
        cmap=cmap,
        legend_kwds={
            'cax': axins, 'orientation': 'horizontal', 'label': label,
            'extend': extend}
    )

    cbar_halo(ax=axins, which='x', linewidth=5, color='0.95')

    rivers_subset.plot(
        ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
        alpha=0.8,
        zorder=5,
        ax=ax)

    hrus_lakes.plot(ec='none', fc='lightblue', zorder=6, ax=ax)

    ax.set_title(title, fontsize=8)

for i, (rr, title) in enumerate(
        zip(
            [p_mean - q_prevah_mean, p_mean - q_mean.sel(model="routed_random")],
            ['Runoff deficit PREVAH', 'Runoff deficit DROP'])
    ):
    ax = axes[1, i]
    ax.set_global()

    norm = mpl.colors.Normalize(vmin=-3, vmax=3)
    cmap = sns.color_palette('viridis', as_cmap=True)
    label = 'Runoff deficit (mm d$^{-1}$)'
    extend = 'both'
    
    axins = ax.inset_axes((0.1, 0.88, 0.25, 0.03))
    rr_subset = rr.load()
    hrus_plot = hrus.copy()
    hrus_plot['val'] = np.nan
    hrus_plot.loc[rr_subset.hru.values, 'val'] = rr_subset.values
    
    im = hrus_plot.plot(
        column='val',
        legend=True,
        ec='k',
        lw=0.1,
        alpha=0.6,
        ax=ax,
        norm=norm,
        cmap='coolwarm_r',
        legend_kwds={
            'cax': axins, 'orientation': 'horizontal', 'label': label,
            'extend': extend}
    )
    cbar_halo(ax=axins, which='x', linewidth=5, color='0.95')
    rivers_subset.plot(
        ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
        alpha=0.8,
        zorder=5,
        ax=ax)
    hrus_lakes.plot(ec='none', fc='lightblue', zorder=6, ax=ax)
    ax.set_title(title, fontsize=8)

fig.savefig('./figures/preva_comparison_rev.png', dpi=300, bbox_inches='tight')

In [ ]:
print(f'Median pre-trained vs. post-trained correlation: {corr_rr_pretraining.median().item():.2f}')
print(f'Median Pre-trained vs. post-trained bias: {bias_rr_pretraining.median().item():.2f} mm d$^{{-1}}$')

## Runoff robustness

In [ ]:
dr = (ds.q_pred_mmd.sel(model='routed_random') - ds_pretrain.q_pred_mmd).compute()
dh = (ds.q_pred_mmd.sel(model='routed_hard') - ds_pretrain.q_pred_mmd).compute()

In [ ]:
crh = xr.corr(dr, dh, dim='time').compute()

In [ ]:
cbrh = xr.corr(dr.mean(dim='time'), dh.mean(dim='time')).compute()

In [ ]:
print('Temporal correlation of runoff delta to PREVAH')

for (c, n) in zip([crh], ['crh']):
    print(f'{n}: {c.mean().item():0.2f} $\pm$ {c.std().item():0.2f}')

print('Spatial correlation of biases towards PREVAH')

for (b, n) in zip([cbrh], ['brh']):
    print(f'{n}: {b.item():0.2f}')

## Retention robustness

In [ ]:
rcrh = xr.corr(ds.R_pred.sel(model='routed_random'), ds.R_pred.sel(model='routed_hard'), dim='time').compute()

In [ ]:
rscrh = xr.corr(ds.R_pred_mean.sel(model='routed_random'), ds.R_pred_mean.sel(model='routed_hard')).compute()

In [ ]:
print('Temporal correlation of learned retention')

for (c, n) in zip([rcrh], ['rcrh']):
    print(f'{n}: {c.mean().item():0.2f} $\pm$ {c.std().item():0.2f}')

print('Spatial correlation of mean learned retention')

for (b, n) in zip([rscrh], ['rscrh']):
    print(f'{n}: {b.item():0.2f}')